# Constraint-Based Testing with Z3

In this exercise you use an **SMT solver** (**Z3**) to turn **logical constraints** on inputs into **concrete test data**—a small taste of *constraint-based* and *solver-backed* testing.

You will:
- Meet a tiny but **safety-flavored** system under test (**SuT**): an **environment alert** rule.
- Try **Z3** in isolation: **satisfiable** models vs **`unsat`**, and reading values from a model.
- **Generate witnesses** for control-flow **path conditions**, run the SuT, and **assert** the expected outcome.
- Finish with **extension TODOs** to go further from the same setup.

**Setup:** install Z3's Python bindings if needed: `pip install z3-solver`. Run cells **from top to bottom**.


In [1]:
# If this import fails in your environment, run: pip install z3-solver
from __future__ import annotations

from z3 import And, Or, Reals, Solver, is_rational_value, sat, simplify, unsat

print("Z3 import OK")


ModuleNotFoundError: No module named 'z3'

## Software under Test (SuT) — a heat–humidity alert

Imagine a **greenhouse**, **livestock barn**, or **data hall** where **heat combined with high humidity** increases risk (heat stress, condensation, or cooling load). A monitor reads **temperature** (`temp_c`, °C) and **relative humidity** (`humidity_pct`, %).

**Function:** `environment_alert(temp_c: float, humidity_pct: float) -> str`

**Intended behavior**
- Return **`"ALERT"`** when it is **hot and humid at once**: `temp_c > 30.0` **and** `humidity_pct >= 70.0`.
- Otherwise return **`"normal"`**.

This matches the lecture's running example: one **compound decision** so we can talk about **path conditions** without extra noise. The next cell is the implementation you will **exercise** with solver-generated inputs.


In [ ]:
def environment_alert(temp_c: float, humidity_pct: float) -> str:
    """Return ALERT when it is hot and humid; otherwise normal."""
    if temp_c > 30.0 and humidity_pct >= 70.0:
        return "ALERT"
    return "normal"


## Z3 in a nutshell

**Z3** is an **SMT** (*Satisfiability Modulo Theories*) **solver**: you give it **first-order constraints** over symbols (here, real arithmetic), and it answers **`sat`** or **`unsat`**.

- **`sat`**: there **exists** an assignment to the symbols that makes **all** asserted constraints true. Z3 can return a **model**—concrete values for those symbols.
- **`unsat`**: the constraints are **jointly impossible**. That is useful in testing when a path or requirement combination is **infeasible** under your assumptions.

Below, `temp_c` and `humidity_pct` in the SuT are Python `float`s; in Z3 we introduce **symbolic** reals `t` and `h` that stand for unknown inputs, then **solve** for them.


In [ ]:
# Symbolic inputs (unknown reals until the solver fixes them)
t, h = Reals("t h")

s = Solver()
# Same shape as the ALERT branch guard in the SuT (strict > and >= as in the code)
s.add(t > 30.0, h >= 70.0)

print("check:", s.check())
if s.check() == sat:
    m = s.model()
    print("model (raw):", m)
else:
    raise SystemExit("expected sat for this toy constraint")


## From a Z3 model to Python `float`s

To **run** `environment_alert`, you need numeric `float`s. For rational values in the model, a small conversion helper is enough for this lab.

Run the cell: it prints one satisfying pair `(t, h)`—any such pair **must** hit the **ALERT** branch by construction of the constraints.


In [ ]:
def z3_real_to_float(val) -> float:
    v = simplify(val)
    if is_rational_value(v):
        return v.numerator_as_long() / v.denominator_as_long()
    raise TypeError(f"Unsupported numeric value in model: {val!r}")


m = s.model()
t_val = z3_real_to_float(m[t])
h_val = z3_real_to_float(m[h])
print(f"t = {t_val!r}, h = {h_val!r}")
print("ALERT guard holds?", (t_val > 30.0) and (h_val >= 70.0))


## When Z3 says `unsat`

If you assert **contradictory** facts, the solver returns **`unsat`**: **no** assignment exists. In testing, that often means an **infeasible path** or **inconsistent requirements** under the assumptions you modeled.


In [ ]:
s2 = Solver()
s2.add(t > 30.0, t < 28.0)  # impossible for a single t
print("check (expected unsat):", s2.check())


## Using Z3 to **target** a branch of the SuT

**Path condition** for the first branch (return `ALERT` as a string):

`temp_c > 30.0` ∧ `humidity_pct >= 70.0`

Steps:
1. Build a `Solver`, add constraints **equivalent** to that guard (using `t`, `h`).
2. If the result is **`sat`**, read `t` and `h` from the model and call `environment_alert(t, h)`.
3. **Assert** the outcome is the string **ALERT**—the input was **designed** to take that branch, not guessed at random.

This is the core idea of **constraint-based test input generation**: the **solver** proposes **witnesses**; the **test** checks runtime behavior.


In [ ]:
def witness_alert() -> tuple[float, float]:
    tt, hh = Reals("t h")
    sol = Solver()
    sol.add(tt > 30.0, hh >= 70.0)
    assert sol.check() == sat
    m = sol.model()
    return z3_real_to_float(m[tt]), z3_real_to_float(m[hh])


a_t, a_h = witness_alert()
assert environment_alert(a_t, a_h) == "ALERT"
print("solver witness:", (a_t, a_h), "->", environment_alert(a_t, a_h))


## `normal` region and an `unsat` **envelope** (from the lecture idea)

**`normal` path:** at least one of the conjuncts fails—formally `¬(t > 30 ∧ h ≥ 70)` which is equivalent to `t ≤ 30 ∨ h < 70`. Z3 can still find a witness.

**Deployment envelope:** suppose sensors in a **product mode** never report `t > 28`. Combining that with the **ALERT** guard can make the whole formula **`unsat`**—there is **no** input that is both **allowed** and **should ALERT** under those assumptions. That is a signal for **requirements review**, not a failing unit test.


In [ ]:
def witness_normal() -> tuple[float, float]:
    tt, hh = Reals("t h")
    sol = Solver()
    sol.add(Or(tt <= 30.0, hh < 70.0))
    assert sol.check() == sat
    m = sol.model()
    return z3_real_to_float(m[tt]), z3_real_to_float(m[hh])


n_t, n_h = witness_normal()
assert environment_alert(n_t, n_h) == "normal"
print("normal witness:", (n_t, n_h), "->", environment_alert(n_t, n_h))

# Envelope: "sensors never above 28C" AND alert guard -> unsat
tt2, hh2 = Reals("t2 h2")
sol3 = Solver()
sol3.add(tt2 <= 28.0)  # operational envelope
sol3.add(tt2 > 30.0, hh2 >= 70.0)  # ALERT guard
print("envelope ∧ ALERT guard:", sol3.check())
assert sol3.check() == unsat


## Extension TODOs (build on the last cells)

1. **Boundary bias:** after asserting the ALERT guard, add **extra** constraints so the witness sits on a **boundary** (e.g. `h == 70.0` or `t` just above `30.0`). Re-run `environment_alert` and keep the assertion.
2. **Second normal witness:** strengthen the normal-region model so humidity is **high** but temperature is **not** (e.g. `h >= 70` **and** `t <= 30`). Explain in a sentence why this must still yield `"normal"`.
3. **Tiny mutation hunt:** change the SuT guard to `>=` vs `>` (or flip `70.0` to `70.0001`) and observe whether your **boundary-tuned** Z3 test **fails** while a random interior point still passes.
4. **(Stretch)** Encode a small **interval** for sensor noise (e.g. `t` in `[t0-0.5, t0+0.5]`) and ask Z3 whether **any** realization can still trigger ALERT.


In [ ]:
# TODO 1 — boundary-biased ALERT witness (fill in `sol_b` constraints)
# tt, hh = Reals("t_b h_b")
# sol_b = Solver()
# sol_b.add( ... ALERT guard ... , ... boundary extras ... )
# ...

# TODO 2 — normal witness with high humidity but t <= 30
# ...

# TODO 3 — optional: duplicate `environment_alert`, mutate the comparison, re-run your boundary test

print("Fill in the TODOs above when you are ready to extend the exercise.")
